# RGB-D → Voxel Reconstruction (Pix2Vox-style)
**Architecture:** Encoder → Decoder → Refiner  
**Input:** RGB-D image (4 channels)  
**Output:** 32×32×32 voxel occupancy grid  
**Dataset:** ModelNet10

## 1. Imports & Device Setup

In [1]:
# Core
import os
import time

# Data/plotting
import matplotlib.pyplot as plt
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Dataset
from modelnet10_rgbd_dataset import make_dataloaders


## 2. Hyperparameters

In [5]:
# ── Data ─────────────────────────────────────────────────────────────
DATA_ROOT   = './ModelNet10'
BATCH_SIZE  = 8
NUM_WORKERS = 4
DEVICE = "cuda"

# ── Model ────────────────────────────────────────────────────────────
VOXEL_SIZE  = 32          # output voxel grid: 32³
IN_CHANNELS = 4           # RGBD (3 colour + 1 depth)

# ── Training ─────────────────────────────────────────────────────────
NUM_EPOCHS      = 50
LR              = 1e-3
LR_STEP         = 20      # halve LR every N epochs
LR_GAMMA        = 0.5
WEIGHT_DECAY    = 1e-4
SAVE_DIR        = './checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

## 3. Data Loaders

In [6]:
train_loader, test_loader = make_dataloaders(
    DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

## 4. Model Architecture

```
RGBD image (B,4,H,W)
      │
  [Encoder]   – ResNet-style 2-D CNN → feature vector
      │
  [Decoder]   – 3-D transposed convolutions → coarse 32³ voxels
      │
  [Refiner]   – lightweight 3-D CNN → refined 32³ voxels
      │
  sigmoid → occupancy probability per voxel
```

In [7]:
# ─────────────────────────────────────────────────────────────────────
# 4-A  Encoder  (2-D CNN on RGBD image)
# ─────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    """Encodes a (B, 4, H, W) RGBD image into a latent vector."""

    def _conv_block(self, in_c, out_c, stride=1):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def __init__(self, in_channels=4, latent_dim=1024):
        super().__init__()
        self.features = nn.Sequential(
            self._conv_block(in_channels,  32, stride=2),   # H/2
            self._conv_block(32,           64, stride=2),   # H/4
            self._conv_block(64,          128, stride=2),   # H/8
            self._conv_block(128,         256, stride=2),   # H/16
            self._conv_block(256,         512, stride=2),   # H/32
        )
        self.pool    = nn.AdaptiveAvgPool2d(1)              # → (B,512,1,1)
        self.fc      = nn.Linear(512, latent_dim)
        self.act     = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)   # (B, 512)
        x = self.act(self.fc(x))      # (B, latent_dim)
        return x


# ─────────────────────────────────────────────────────────────────────
# 4-B  Decoder  (3-D transposed convolutions)
# ─────────────────────────────────────────────────────────────────────
class Decoder(nn.Module):
    """Maps a latent vector → coarse (B,1,V,V,V) voxel grid."""

    def _deconv_block(self, in_c, out_c, k=4, s=2, p=1):
        return nn.Sequential(
            nn.ConvTranspose3d(in_c, out_c, k, stride=s, padding=p, bias=False),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True),
        )

    def __init__(self, latent_dim=1024, voxel_size=32):
        super().__init__()
        self.voxel_size = voxel_size
        # Project latent → 256 channels in a 2³ seed volume
        self.fc      = nn.Linear(latent_dim, 256 * 2 * 2 * 2)
        self.network = nn.Sequential(
            self._deconv_block(256, 128),   # 2 → 4
            self._deconv_block(128,  64),   # 4 → 8
            self._deconv_block( 64,  32),   # 8 → 16
            self._deconv_block( 32,  16),   # 16 → 32
            nn.ConvTranspose3d(16, 1, 1),   # channel → 1
            nn.Sigmoid(),
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 2, 2, 2)
        return self.network(x)             # (B,1,V,V,V)


# ─────────────────────────────────────────────────────────────────────
# 4-C  Refiner  (lightweight 3-D CNN on coarse voxels)
# ─────────────────────────────────────────────────────────────────────
class Refiner(nn.Module):
    """Sharpens the coarse voxel grid with 3-D convolutions."""

    def _res_block(self, channels):
        return nn.Sequential(
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm3d(channels),
        )

    def __init__(self):
        super().__init__()
        self.conv_in  = nn.Conv3d(1, 32, 3, padding=1)
        self.res1     = self._res_block(32)
        self.res2     = self._res_block(32)
        self.conv_out = nn.Conv3d(32, 1, 1)
        self.relu     = nn.ReLU(inplace=True)

    def forward(self, coarse):
        x  = self.relu(self.conv_in(coarse))      # (B,32,V,V,V)
        r1 = self.relu(self.res1(x) + x)
        r2 = self.relu(self.res2(r1) + r1)
        return torch.sigmoid(self.conv_out(r2))   # (B,1,V,V,V)


# ─────────────────────────────────────────────────────────────────────
# 4-D  Full model
# ─────────────────────────────────────────────────────────────────────
class RGBDToVoxel(nn.Module):
    """
    End-to-end RGBD → Voxel model.
    Returns (coarse, refined) during training for dual loss;
    only refined at inference.
    """

    def __init__(self, in_channels=4, latent_dim=1024, voxel_size=32):
        super().__init__()
        self.encoder = Encoder(in_channels=in_channels, latent_dim=latent_dim)
        self.decoder = Decoder(latent_dim=latent_dim, voxel_size=voxel_size)
        self.refiner = Refiner()

    def forward(self, x):
        z      = self.encoder(x)        # (B, latent_dim)
        coarse = self.decoder(z)        # (B,1,V,V,V)
        refined = self.refiner(coarse)  # (B,1,V,V,V)
        return coarse, refined


# ── Build & summarise ────────────────────────────────────────────────
model = RGBDToVoxel(
    in_channels=IN_CHANNELS,
    latent_dim=1024,
    voxel_size=VOXEL_SIZE,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')
print(model)

Trainable parameters: 7,091,922
RGBDToVoxel(
  (encoder): Encoder(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2d(4, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (1): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (2): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (3): Sequential(
        (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), 

## 5. Loss Function & Optimiser

In [8]:
# ── Weighted Binary Cross-Entropy (handles voxel sparsity) ───────────
class WeightedBCELoss(nn.Module):
    """
    BCE with a higher weight on occupied voxels to counter class imbalance
    (most voxels are empty for typical object shapes).
    """
    def __init__(self, pos_weight=2.0):
        super().__init__()
        self.pos_weight = pos_weight

    def forward(self, pred, target):
        # pred, target: (B,1,V,V,V)  values in [0,1]
        weight = target * (self.pos_weight - 1) + 1   # occupied → pos_weight, empty → 1
        return F.binary_cross_entropy(pred, target, weight=weight)


criterion = WeightedBCELoss(pos_weight=2.0)

optimizer = optim.Adam(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

print('Loss      :', criterion)
print('Optimiser :', optimizer)
print('Scheduler :', scheduler)

Loss      : WeightedBCELoss()
Optimiser : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)
Scheduler : <torch.optim.lr_scheduler.StepLR object at 0x000001A95148A4A0>


## 6. Helper: Batch Unpacking

Adjust `get_rgbd_and_voxel` if your dataset uses different key names.

In [9]:
def get_rgbd_and_voxel(batch, device):
    """
    Extract RGBD image and voxel target from a loader batch.
    Handles common key conventions from ModelNet10RGBDDataset.

    Returns
    -------
    rgbd  : (B, 4, H, W)  float32
    voxel : (B, 1, V, V, V)  float32  (binary occupancy)
    """
    # ── RGB ──────────────────────────────────────────────────────────
    rgb_key = next(k for k in ('rgb', 'image', 'color') if k in batch)
    rgb = batch[rgb_key].float()           # (B, 3, H, W)  or  (B, H, W, 3)
    if rgb.shape[1] != 3:                  # channel-last → channel-first
        rgb = rgb.permute(0, 3, 1, 2)
    rgb = rgb / 255.0 if rgb.max() > 1.0 else rgb   # normalise to [0,1]

    # ── Depth ────────────────────────────────────────────────────────
    dep_key = next(k for k in ('depth',) if k in batch)
    dep = batch[dep_key].float()           # (B, H, W)  or (B, 1, H, W)
    if dep.dim() == 3:                     # → (B,1,H,W)
        dep = dep.unsqueeze(1)
    # Normalise depth to [0,1] using per-batch min-max
    d_min = dep.flatten(1).min(1).values.view(-1,1,1,1)
    d_max = dep.flatten(1).max(1).values.view(-1,1,1,1)
    dep = (dep - d_min) / (d_max - d_min + 1e-8)

    # ── Concatenate RGBD ─────────────────────────────────────────────
    rgbd = torch.cat([rgb, dep], dim=1).to(device)   # (B, 4, H, W)

    # ── Voxel target ─────────────────────────────────────────────────
    vox_key = next(k for k in ('voxel', 'voxels', 'voxel_grid') if k in batch)
    voxel = batch[vox_key].float()         # (B, V, V, V)  or (B,1,V,V,V)
    if voxel.dim() == 4:
        voxel = voxel.unsqueeze(1)         # → (B,1,V,V,V)
    # Resize if needed to VOXEL_SIZE
    if voxel.shape[-1] != VOXEL_SIZE:
        voxel = F.interpolate(
            voxel, size=(VOXEL_SIZE,) * 3,
            mode='trilinear', align_corners=False
        )
    voxel = (voxel > 0.5).float().to(device)   # binarise

    return rgbd, voxel

## 7. Metric: Intersection-over-Union (IoU)

In [10]:
@torch.no_grad()
def voxel_iou(pred, target, threshold=0.5):
    """
    Compute mean IoU over a batch.
    pred, target : (B,1,V,V,V) float tensors
    """
    pred_bin = (pred > threshold).float()
    inter    = (pred_bin * target).flatten(1).sum(1)
    union    = (pred_bin + target).clamp(max=1).flatten(1).sum(1)
    iou      = inter / (union + 1e-8)
    return iou.mean().item()

## 8. Training Loop

In [11]:
history = {
    'train_loss': [], 'train_iou': [],
    'val_loss'  : [], 'val_iou'  : [],
}

best_val_iou = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # ── TRAIN ────────────────────────────────────────────────────────
    model.train()
    train_loss, train_iou = 0.0, 0.0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [train]', leave=False):
        rgbd, voxel = get_rgbd_and_voxel(batch, DEVICE)

        optimizer.zero_grad()
        coarse, refined = model(rgbd)

        # Dual loss: coarse gets 0.3 weight, refined gets 0.7
        loss = 0.3 * criterion(coarse, voxel) + 0.7 * criterion(refined, voxel)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_iou  += voxel_iou(refined, voxel)

    train_loss /= len(train_loader)
    train_iou  /= len(train_loader)

    # ── VALIDATE ─────────────────────────────────────────────────────
    model.eval()
    val_loss, val_iou = 0.0, 0.0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [val]  ', leave=False):
            rgbd, voxel  = get_rgbd_and_voxel(batch, DEVICE)
            _, refined   = model(rgbd)
            val_loss    += criterion(refined, voxel).item()
            val_iou     += voxel_iou(refined, voxel)

    val_loss /= len(test_loader)
    val_iou  /= len(test_loader)

    scheduler.step()

    # ── Logging ──────────────────────────────────────────────────────
    elapsed = time.time() - t0
    print(
        f'Epoch {epoch:03d}/{NUM_EPOCHS}  '
        f'Train loss={train_loss:.4f}  IoU={train_iou:.4f}  |  '
        f'Val   loss={val_loss:.4f}  IoU={val_iou:.4f}  '
        f'[{elapsed:.1f}s]  lr={scheduler.get_last_lr()[0]:.2e}'
    )

    history['train_loss'].append(train_loss)
    history['train_iou'].append(train_iou)
    history['val_loss'].append(val_loss)
    history['val_iou'].append(val_iou)

    # ── Checkpoint (best model) ───────────────────────────────────────
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        ckpt_path = os.path.join(SAVE_DIR, 'best_model.pth')
        torch.save({
            'epoch'     : epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_iou'   : val_iou,
            'val_loss'  : val_loss,
        }, ckpt_path)
        print(f'  ✓ New best model saved (val IoU={best_val_iou:.4f})')

print(f'\nTraining complete. Best val IoU: {best_val_iou:.4f}')

KeyboardInterrupt: 

## 9. Training Curves

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_loss'], label='Train Loss')
ax1.plot(epochs, history['val_loss'],   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Weighted BCE Loss')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(True)

ax2.plot(epochs, history['train_iou'], label='Train IoU')
ax2.plot(epochs, history['val_iou'],   label='Val IoU')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('IoU')
ax2.set_title('IoU Curve'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Qualitative Visualisation

In [ ]:
# Load best checkpoint
ckpt = torch.load(os.path.join(SAVE_DIR, 'best_model.pth'), map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Loaded best model from epoch {ckpt['epoch']}  (val IoU={ckpt['val_iou']:.4f})")

# Grab one test batch
batch            = next(iter(test_loader))
rgbd, voxel_gt   = get_rgbd_and_voxel(batch, DEVICE)

with torch.no_grad():
    _, voxel_pred = model(rgbd)

# Show first sample
idx   = 0
gt_np = voxel_gt[idx, 0].cpu().numpy()                      # (V,V,V)
pd_np = (voxel_pred[idx, 0] > 0.5).float().cpu().numpy()   # (V,V,V)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
slice_ids = [VOXEL_SIZE // 4, VOXEL_SIZE // 2, 3 * VOXEL_SIZE // 4]

for col, sl in enumerate(slice_ids):
    axes[0, col].imshow(gt_np[sl], cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'GT  slice z={sl}')
    axes[0, col].axis('off')

    axes[1, col].imshow(pd_np[sl], cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'Pred slice z={sl}')
    axes[1, col].axis('off')

plt.suptitle(
    f'IoU = {voxel_iou(voxel_pred[idx:idx+1], voxel_gt[idx:idx+1]):.4f}',
    fontsize=14
)
plt.tight_layout()
plt.show()

## 11. Inference on a Single Sample

In [ ]:
from modelnet10_rgbd_dataset import ModelNet10RGBDDataset
import torchvision.transforms.functional as TF

def predict_voxel(rgb_tensor, depth_tensor, model, device=DEVICE):
    """
    Run inference on a single RGBD sample.

    Parameters
    ----------
    rgb_tensor   : (3, H, W)  float in [0,1]
    depth_tensor : (1, H, W)  float (raw depth values)
    """
    model.eval()

    d_min  = depth_tensor.min()
    d_max  = depth_tensor.max()
    depth  = (depth_tensor - d_min) / (d_max - d_min + 1e-8)

    rgbd   = torch.cat([rgb_tensor, depth], dim=0).unsqueeze(0).to(device)  # (1,4,H,W)

    with torch.no_grad():
        _, refined = model(rgbd)   # (1,1,V,V,V)

    return (refined[0, 0] > 0.5).cpu().numpy()   # (V,V,V) bool numpy


# Example usage
sample  = ModelNet10RGBDDataset('./ModelNet10', split='test')[0]

# Adjust key names to match what your dataset returns
rgb_key = next(k for k in ('rgb', 'image', 'color') if k in sample)
dep_key = 'depth'

rgb   = torch.tensor(sample[rgb_key]).float()
depth = torch.tensor(sample[dep_key]).float()
if rgb.shape[0] != 3:   rgb   = rgb.permute(2,0,1)
if depth.dim() == 2:    depth = depth.unsqueeze(0)
if rgb.max() > 1:       rgb   = rgb / 255.0

vox = predict_voxel(rgb, depth, model)
print('Output voxel grid shape:', vox.shape)
print('Occupied voxels:', vox.sum(), '/', vox.size)